<a href="https://colab.research.google.com/github/AI-is-out-there/FDA-AI-products-as-graph/blob/main/code/Graph-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import subprocess
import sys
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Installing required packages...")
for pkg in ["pandas", "networkx", "plotly", "python-dateutil", "scikit-network"]:
    install(pkg)

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("\n" + "="*80)
print("FDA MEDICAL DEVICE NETWORK ANALYSIS")
print("="*80)
print("\n1. Loading and preparing data...")

# Read the CSV file
df = pd.read_csv('/content/dataset-FDA-med-graph.csv', delimiter=';')

# Parse dates
df['Date of Final Decision'] = pd.to_datetime(df['Date of Final Decision'], format='%d.%m.%Y')
df['Month_Year'] = df['Date of Final Decision'].dt.strftime('%b %Y')

# --- NEW: Explicitly handle 'Unknown' and 'External' as missing values ---
# If 'Unknown' or 'External' are literal strings in the CSV, convert them to NaN
df['Primary Product Code'] = df['Primary Product Code'].replace('Unknown', np.nan)
df['Panel (Lead)'] = df['Panel (Lead)'].replace('External', np.nan)
# --- END NEW ---

# Data cleaning
# Now, dropna will correctly remove rows where 'Primary Product Code' or 'Panel (Lead)' were originally
# 'Unknown' or 'External' strings (now NaN), as well as any other genuine NaNs in these columns.
df = df.dropna(subset=['Submission Number', 'Device', 'Panel (Lead)', 'Primary Product Code'])

print(f"✓ Loaded {len(df)} records")
print(f"  Date range: {df['Date of Final Decision'].min().date()} to {df['Date of Final Decision'].max().date()}")
print(f"  Unique panels: {df['Panel (Lead)'].nunique()}")
print(f"  Unique product codes: {df['Primary Product Code'].nunique()}")
print(f"  Unique companies: {df['Company'].nunique()}")

# ============================================================================
# 2. BUILD NETWORK GRAPH (IMPROVED ALGORITHM)
# ============================================================================
print("\n2. Building network graph...")

G = nx.DiGraph()

# First pass: Add all nodes from the dataset
for idx, row in df.iterrows():
    submission = row['Submission Number']
    G.add_node(submission,
               device=row['Device'],
               company=row['Company'],
               date_decision=row['Date of Final Decision'],
               month_year=row['Month_Year'],
               panel=row['Panel (Lead)'],
               product_code=row['Primary Product Code'])

# Second pass: Add edges ONLY if both submission and predicate exist in dataset
edge_count = 0
predicate_not_found = 0

for idx, row in df.iterrows():
    submission = row['Submission Number']
    predicate = row['Predicate Device']

    # Only create edge if predicate exists in our dataset AND has a value
    if pd.notna(predicate) and predicate.strip():
        # Check if predicate exists as a node in our graph
        if predicate in G.nodes():
            # Only add edge if it doesn't already exist
            if not G.has_edge(predicate, submission):
                G.add_edge(predicate, submission)
                edge_count += 1
        else:
            predicate_not_found += 1

print(f"✓ Created graph with {G.number_of_nodes()} nodes and {edge_count} edges")
print(f"  Predicates not found in dataset (skipped): {predicate_not_found}")

# ============================================================================
# 3. NETWORK STATISTICS
# ============================================================================
print("\n3. Calculating network statistics...")

in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degree = {node: in_degrees[node] + out_degrees[node] for node in G.nodes()}

# Find top nodes
top_referenced = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
top_referencing = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nMost Referenced Devices (Top 5):")
for node, degree in top_referenced:
    if degree > 0:
        device = G.nodes[node].get('device', node)
        print(f"  {degree:2d}x - {device[:50]}")

print("\nMost Referencing Devices (Top 5):")
for node, degree in top_referencing:
    if degree > 0:
        device = G.nodes[node].get('device', node)
        print(f"  {degree:2d}x - {device[:50]}")

# ============================================================================
# 4. CREATE HIERARCHICAL LAYOUT
# ============================================================================
print("\n4. Calculating hierarchical layout...")

# Group by panel and product code
panel_groups = defaultdict(lambda: defaultdict(list))
for node in G.nodes():
    # Use .get with a default for panel and product_code, as they might now be NaN
    panel = G.nodes[node].get('panel', 'Unknown') # Default to 'Unknown' if NaN after conversion
    product_code = G.nodes[node].get('product_code', 'Unknown') # Default to 'Unknown' if NaN after conversion

    # Handle NaN values for grouping - convert them to a consistent string representation if needed for keys
    panel_key = str(panel) if pd.isna(panel) else panel
    product_code_key = str(product_code) if pd.isna(product_code) else product_code

    panel_groups[panel_key][product_code_key].append(node)

# Create timeline mapping
unique_dates = sorted([d for d in df['Date of Final Decision'].unique() if pd.notna(d)])
date_mapping = {date: idx for idx, date in enumerate(unique_dates)}

# Calculate positions
pos = {}
panel_list = sorted(panel_groups.keys())

for panel_idx, panel in enumerate(panel_list):
    product_codes = sorted(panel_groups[panel].keys())
    y_offset_panel = panel_idx * 5

    for prod_idx, product_code in enumerate(product_codes):
        nodes_in_group = panel_groups[panel][product_code]
        y_base = y_offset_panel + prod_idx * 1

        for node_idx, node in enumerate(nodes_in_group):
            # Get x position from date
            node_data = G.nodes[node]
            date = node_data.get('date_decision')

            if date and date in date_mapping:
                x_pos = date_mapping[date]
            else:
                x_pos = len(unique_dates) / 2 # Fallback if date is missing or not in unique_dates

            y_pos = y_base + (node_idx * 0.1)
            pos[node] = (x_pos, y_pos)

print(f"✓ Layout calculated for {len(pos)} nodes")

# ============================================================================
# 5. PREPARE VISUALIZATION DATA
# ============================================================================
print("\n5. Preparing visualization...")

# Prepare edge data
edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

# Prepare node data
node_x, node_y = [], []
node_text, node_colors = [], []
node_sizes, node_hover = [], []

# Color mapping
unique_panels = list(panel_groups.keys()) # Now includes 'Unknown' or 'External' if present
colors = px.colors.qualitative.Set3 if len(unique_panels) <= 12 else px.colors.qualitative.Light24
panel_color_map = {panel: colors[i % len(colors)] for i, panel in enumerate(unique_panels)}

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    node_data = G.nodes[node]
    device = node_data.get('device', node)
    company = node_data.get('company', 'Unknown')
    date = node_data.get('date_decision')

    # Get panel and product_code, providing 'Unknown' as fallback if NaN
    panel = node_data.get('panel', 'Unknown')
    product_code = node_data.get('product_code', 'Unknown')

    month_year = node_data.get('month_year', 'Unknown')

    # Node label: Device name (truncated) + Month/Year
    if len(device) > 25:
        label = f"{device[:22]}... ({month_year})"
    else:
        label = f"{device} ({month_year})"
    node_text.append(label)

    # Color by panel
    node_colors.append(panel_color_map.get(str(panel), colors[0])) # Ensure panel key is string

    # Size by network degree
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)
    size = 15 + (in_deg + out_deg) * 3
    node_sizes.append(size)

    # Hover information
    date_str = date.strftime('%d.%m.%Y') if pd.notna(date) else 'Unknown'
    hover = f"<b>{device}</b><br>" \
            f"<b>ID:</b> {node}<br>" \
            f"<b>Company:</b> {company}<br>" \
            f"<b>Date:</b> {date_str}<br>" \
            f"<b>Panel:</b> {panel}<br>" \
            f"<b>Product Code:</b> {product_code}<br>" \
            f"<b>Referenced by:</b> {in_deg} | " \
            f"<b>References:</b> {out_deg}"
    node_hover.append(hover)

# ============================================================================
# 6. CREATE MAIN VISUALIZATION
# ============================================================================
print("\n6. Creating interactive visualization...")

fig = go.Figure()

# Add edges with gradient opacity
fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='rgba(125,125,125,0.15)'),
    hoverinfo='skip',
    showlegend=False,
    name='Predicate Links'
))

# Add nodes
fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    marker=dict(
        size=node_sizes,
        color=node_colors,
        opacity=0.85,
        line=dict(width=2, color='white'),
        symbol='circle'
    ),
    text=node_text,
    textposition='top center',
    textfont=dict(size=7, color='black'),
    hovertext=node_hover,
    hoverinfo='text',
    showlegend=False,
    name='Devices'
))

# Add panel legend
for i, panel in enumerate(unique_panels):
    color = panel_color_map[panel]
    count = len([n for n in G.nodes() if G.nodes[n].get('panel') == (None if panel == 'nan' else panel)]) # Handle 'nan' string from key conversion
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color=color, line=dict(width=2, color='white')),
        name=f'{panel} ({count})',
        showlegend=True
    ))

# Update layout
x_labels = [d.strftime('%b %Y') for d in unique_dates]
x_ticks = list(range(len(unique_dates)))

fig.update_layout(
    title={
        'text': '<b>FDA Medical Device Network - Timeline & Hierarchical View</b><br>'
                '<sub>Nodes: Devices | Edges: Predicate References | Size: Network Centrality</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#1a1a1a'}
    },
    showlegend=True,
    legend=dict(
        title=dict(text='<b>Panels (Count)</b>', font=dict(size=12, color='black')),
        yanchor='top',
        y=0.99,
        xanchor='right',
        x=0.99,
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#333333',
        borderwidth=2,
        font=dict(size=10, color='black')
    ),
    hovermode='closest',
    margin=dict(b=120, l=80, r=150, t=140),
    xaxis=dict(
        title='<b>Timeline - Date of Final Decision</b>',
        ticktext=x_labels,
        tickvals=x_ticks,
        tickangle=-45,
        showgrid=False,  # NO GRID
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='#333333'
    ),
    yaxis=dict(
        title='<b>Hierarchical Grouping (Panel → Product Code)</b>',
        showgrid=False,  # NO GRID
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='#333333'
    ),
    plot_bgcolor='white',  # WHITE BACKGROUND
    paper_bgcolor='white',  # WHITE BACKGROUND
    width=1800,
    height=1100,
    font=dict(family='Arial, sans-serif', size=11, color='#1a1a1a'),
    dragmode='zoom'  # Better zoom interaction
)

fig.show()

# Save as interactive HTML file
print("\nSaving main graph as interactive HTML file...")
html_file_path = '/content/fda_network_graph_main.html'
fig.write_html(html_file_path)
print(f"✓ Saved to: {html_file_path}")

# ============================================================================
# 7. CREATE SUMMARY STATISTICS VISUALIZATION
# ============================================================================
print("\n7. Creating summary statistics...")

# Create a summary statistics figure
summary_data = []

# Panel distribution
panel_counts = {}
for panel in unique_panels:
    count = len([n for n in G.nodes() if G.nodes[n].get('panel') == (None if panel == 'nan' else panel)]) # Handle 'nan' string
    panel_counts[panel] = count
    summary_data.append({'Category': 'Panel', 'Value': panel, 'Count': count})

# Product code distribution (top 10)
product_counts = Counter()
for node in G.nodes():
    product = G.nodes[node].get('product_code', 'Unknown')
    product_counts[product] += 1

# Device statistics over time
time_counts = Counter()
for date in unique_dates:
    # Count submissions where 'Date of Final Decision' matches and the row was not dropped
    count = len(df[df['Date of Final Decision'] == date])
    time_counts[date] = count

# Create subplots
fig_stats = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Devices by Panel', 'Top 10 Product Codes',
                    'Submissions Over Time', 'Network Degree Distribution'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'histogram'}]]
)

# Panel distribution
panels = list(panel_counts.keys())
panel_vals = list(panel_counts.values())
fig_stats.add_trace(
    go.Bar(x=panels, y=panel_vals, name='Devices',
           marker=dict(color=px.colors.qualitative.Set2[:len(panels)])),
    row=1, col=1
)

# Top product codes
top_products = product_counts.most_common(10)
prod_names = [str(p[0]) for p in top_products] # Convert to string for plotting if NaN is present
prod_vals = [p[1] for p in top_products]
fig_stats.add_trace(
    go.Bar(x=prod_names, y=prod_vals, name='Count',
           marker=dict(color='indianred')),
    row=1, col=2
)

# Time series
dates = sorted(time_counts.keys())
time_vals = [time_counts[d] for d in dates]
date_labels = [d.strftime('%m/%d') for d in dates]
fig_stats.add_trace(
    go.Scatter(x=date_labels, y=time_vals, mode='lines+markers',
               name='Submissions', fill='tozeroy',
               line=dict(color='steelblue')),
    row=2, col=1
)

# Degree distribution
degrees = [total_degree[node] for node in G.nodes()]
fig_stats.add_trace(
    go.Histogram(x=degrees, nbinsx=30, name='Degree',
                 marker=dict(color='mediumseagreen')),
    row=2, col=2
)

# Update layout
fig_stats.update_xaxes(title_text="Panel", row=1, col=1)
fig_stats.update_xaxes(title_text="Product Code", row=1, col=2)
fig_stats.update_xaxes(title_text="Date", row=2, col=1)
fig_stats.update_xaxes(title_text="Node Degree", row=2, col=2)

fig_stats.update_yaxes(title_text="Count", row=1, col=1)
fig_stats.update_yaxes(title_text="Count", row=1, col=2)
fig_stats.update_yaxes(title_text="Submissions", row=2, col=1)
fig_stats.update_yaxes(title_text="Frequency", row=2, col=2)

fig_stats.update_layout(
    title_text='<b>FDA Medical Device Dataset - Summary Statistics</b>',
    height=800,
    width=1600,
    showlegend=False
)

fig_stats.show()

# ============================================================================
# 8. EXPORT DATA
# ============================================================================
print("\n8. Exporting detailed data...")

# Create nodes dataframe
nodes_list = []
for node in G.nodes():
    node_data = G.nodes[node]
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)

    nodes_list.append({
        'Submission Number': node,
        'Device': node_data.get('device', ''),
        'Company': node_data.get('company', ''),
        'Date of Final Decision': node_data.get('date_decision'),
        'Month Year': node_data.get('month_year', ''),
        'Panel': node_data.get('panel', 'Unknown'), # Default to 'Unknown' if NaN
        'Product Code': node_data.get('product_code', 'Unknown'), # Default to 'Unknown' if NaN
        'In-Degree (Referenced By)': in_deg,
        'Out-Degree (References)': out_deg,
        'Total Degree': in_deg + out_deg
    })

nodes_export_df = pd.DataFrame(nodes_list)

# Create edges dataframe
edges_list = []
for source, target in G.edges():
    source_data = G.nodes[source]
    target_data = G.nodes[target]

    edges_list.append({
        'Source Submission': source,
        'Source Device': source_data.get('device', ''),
        'Source Company': source_data.get('company', ''),
        'Target Submission': target,
        'Target Device': target_data.get('device', ''),
        'Target Company': target_data.get('company', ''),
        'Relationship Type': 'Predicate Device Reference'
    })

edges_export_df = pd.DataFrame(edges_list)

print(f"\n✓ Exported {len(nodes_export_df)} nodes and {len(edges_export_df)} edges")
print(f"\nNodes DataFrame (first 5 rows):")
print(nodes_export_df.head())
print(f"\nEdges DataFrame (first 5 rows):")
print(edges_export_df.head())

# ============================================================================
# 9. NETWORK ANALYSIS SUMMARY
# ============================================================================
print("\n" + "="*80)
print("NETWORK ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 Graph Statistics:")
print(f"  • Total Nodes: {G.number_of_nodes()}")
print(f"  • Total Edges: {G.number_of_edges()}")
print(f"  • Network Density: {nx.density(G):.4f}")
print(f"  • Average In-Degree: {np.mean(list(in_degrees.values())):.2f}")
print(f"  • Average Out-Degree: {np.mean(list(out_degrees.values())):.2f}")

print(f"\n🏥 Panel Distribution:")
# Iterate through unique_panels (which might contain 'nan' string for missing values)
for panel in panel_list:
    # Count nodes whose 'panel' attribute matches the current panel (or is None if panel is 'nan' string)
    count = len([n for n in G.nodes() if G.nodes[n].get('panel') == (None if panel == 'nan' else panel)])
    pct = (count / G.number_of_nodes()) * 100
    print(f"  • {panel}: {count} devices ({pct:.1f}%)")

print(f"\n🎯 Product Code Statistics:")
print(f"  • Unique Product Codes: {len(product_counts)}")
print(f"  • Most Common: {product_counts.most_common(3)}")

print(f"\n📅 Timeline Statistics:")
# Ensure unique_dates is not empty before accessing elements
if unique_dates:
    print(f"  • Date Range: {unique_dates[0].date()} to {unique_dates[-1].date()}")
    days_covered = (unique_dates[-1] - unique_dates[0]).days
    print(f"  • Days Covered: {days_covered}")
    # Avoid division by zero if no days covered
    avg_submissions_per_day = len(df) / max(days_covered, 1) if days_covered > 0 else 0
    print(f"  • Average Submissions per Day: {avg_submissions_per_day:.2f}")
else:
    print("  • No dates found for timeline statistics.")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE!")
print("="*80)

Installing required packages...

FDA MEDICAL DEVICE NETWORK ANALYSIS

1. Loading and preparing data...
✓ Loaded 1302 records
  Date range: 2001-03-15 to 2025-09-29
  Unique panels: 17
  Unique product codes: 132
  Unique companies: 650

2. Building network graph...
✓ Created graph with 1302 nodes and 647 edges
  Predicates not found in dataset (skipped): 655

3. Calculating network statistics...

Most Referenced Devices (Top 5):
   1x - SMART PCFD
   1x - Ligence Heart
   1x - Brain WMH
   1x - uMR 680
   1x - SwiftSight-Brain

Most Referencing Devices (Top 5):
  10x - NeuroQuant
   9x - BriefCase
   8x - BriefCase
   8x - AccuContour
   6x - HealthPNX

4. Calculating hierarchical layout...
✓ Layout calculated for 1302 nodes

5. Preparing visualization...

6. Creating interactive visualization...



Saving main graph as interactive HTML file...
✓ Saved to: /content/fda_network_graph_main.html

7. Creating summary statistics...



8. Exporting detailed data...

✓ Exported 1302 nodes and 647 edges

Nodes DataFrame (first 5 rows):
  Submission Number                       Device  \
0           K252054  SpineAR SNAP (SyncAR Spine)   
1           K250023                   SMART PCFD   
2           K252105                Ligence Heart   
3           K251527                    Brain WMH   
4           K252371                      uMR 680   

                                        Company Date of Final Decision  \
0                        Surgical Theater, Inc.             2025-09-29   
1                                    Disior Ltd             2025-09-29   
2                                  Ligence, UAB             2025-09-26   
3                                    Quantib BV             2025-09-25   
4  Shanghai United Imaging Healthcare Co., Ltd.             2025-09-25   

  Month Year      Panel Product Code  In-Degree (Referenced By)  \
0   Sep 2025  Neurology          SBF                          0   
1   Sep